# BÁO CÁO PHÂN TÍCH DỮ LIỆU BẰNG SQL TRÊN DUCKDB (SQL02)

Notebook này tổng hợp toàn bộ các câu lệnh SQL phục vụ cho các câu hỏi nghiên cứu (**Research Questions - RQs**), đồng thời thực thi và xuất các bảng kết quả ra thư mục `report/` dưới dạng file CSV.

## 1. Khởi tạo kết nối DuckDB và Nạp Dữ liệu

In [ ]:
import os
import duckdb
import pandas as pd

os.makedirs('report', exist_ok=True)
os.makedirs('sql', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

db_path = 'data/processed/ady.duckdb'
con = duckdb.connect(db_path)

print("Đã kết nối DuckDB thành công!")

## 2. Chi tiết các câu lệnh SQL theo Research Questions (RQs)

### RQ1: Phân tích phân bố sự cố theo thời gian và nhóm phương tiện (UAS vs Non-UAS)
- **Mục tiêu:** Đánh giá quy luật biến động theo năm và so sánh tỷ lệ sự cố giữa máy bay không người lái (UAS) và các loại máy bay truyền thống.

#### Query 1.1: Thống kê số lượng báo cáo theo năm và phân loại nhóm UAS

In [ ]:
q1_1 = """
-- Link to RQ1: Phân tích xu hướng thời gian giữa UAS và Non-UAS
SELECT
    year AS year,
    is_uas AS is_uas,
    COUNT(*) AS n
FROM r
GROUP BY 1, 2
ORDER BY 1, 2;
"""
df_q1_1 = con.execute(q1_1).df()
df_q1_1

#### Query 1.2: Phân bố giai đoạn bay (Flight Phase) và tỷ trọng % trong từng nhóm bằng Window Function

In [ ]:
q1_2 = """
-- Link to RQ1: Tỷ trọng giai đoạn bay xảy ra sự cố giữa các nhóm phương tiện
SELECT
    is_uas AS is_uas,
    flight_phase AS flight_phase,
    COUNT(*) AS n,
    COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY is_uas) AS pct
FROM r
GROUP BY 1, 2
ORDER BY 1, pct DESC;
"""
df_q1_2 = con.execute(q1_2).df()
df_q1_2

### RQ1 & RQ2: Phân tích thuộc tính văn bản (Narrative) & Loại sự cố (Target Class)
- **Mục tiêu:** Thống kê tần suất các nhóm nhãn sự cố ($y$) và độ dài trung bình đoạn mô tả văn bản (`narrative`), cung cấp đặc trưng cho các mô hình phân loại đa phương thức (Multimodal) và mô hình tóm tắt nhỏ (SLM).

#### Query 1.3: Thống kê nhãn sự cố và độ dài văn bản mô tả trung bình

In [ ]:
q1_3 = """
-- Link to RQ1 & RQ2: Tần suất nhãn mục tiêu và độ dài văn bản Narrative
SELECT
    y AS y,
    COUNT(*) AS n,
    AVG(length(narrative)) AS len
FROM r
GROUP BY 1
ORDER BY n DESC;
"""
df_q1_3 = con.execute(q1_3).df()
df_q1_3

#### Query 1.4: Phân tích rủi ro theo Vùng trời (Airspace) và Điều kiện thời tiết (Weather)

In [ ]:
q1_4 = """
-- Link to RQ1: Mối liên hệ giữa vùng trời, thời tiết và loại sự cố
SELECT
    airspace AS airspace,
    weather AS weather,
    COUNT(*) AS total_incidents,
    SUM(CASE WHEN is_uas = TRUE THEN 1 ELSE 0 END) AS uas_count
FROM r
GROUP BY 1, 2
ORDER BY total_incidents DESC
LIMIT 10;
"""
df_q1_4 = con.execute(q1_4).df()
df_q1_4

### RQ2 & RQ3: Tạo bảng đặc trưng (Feature Table) & Xử lý biến trượt/trễ (Lag & Moving Average)
- **Mục tiêu:** Sử dụng Window Function để tổng hợp thuộc tính cấu trúc và chuỗi thời gian, xuất bảng `feat` chuyển giao sang giai đoạn huấn luyện mô hình.

#### Query 2.1: Tạo bảng đặc trưng tổng hợp (Feature Table `feat`)

In [ ]:
q2_1 = """
-- Link to RQ2 & RQ3: Trích xuất các thuộc tính cấu trúc và văn bản cho Model Training
CREATE OR REPLACE TABLE feat AS
SELECT
    id AS id,
    year AS year,
    is_uas AS is_uas,
    y AS y,
    narrative AS narrative,
    flight_phase AS flight_phase,
    altitude AS altitude,
    light AS light,
    mission AS mission,
    weather AS weather,
    airspace AS airspace,
    aircraft_type AS aircraft_type
FROM r;
"""
con.execute(q2_1)
print("Đã tạo thành công bảng 'feat'!")

#### Query 2.2: Tính chỉ số trung bình trượt 7 kỳ (Moving Average) bằng Window Function

In [ ]:
q2_2 = """
-- Link to RQ1 & RQ3: Tạo đặc trưng trung bình trượt thời gian theo vùng trời
SELECT
    id,
    year,
    airspace,
    AVG(length(narrative)) OVER (
        PARTITION BY airspace
        ORDER BY year
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS narrative_len_ma7
FROM r;
"""
df_q2_2 = con.execute(q2_2).df()
df_q2_2.head()

## 3. Bộ câu truy vấn SQL tham khảo (STATS19 London Collision Data)
*Phần này dành cho trường hợp phân tích bộ dữ liệu tai nạn giao thông London.*

In [ ]:
stats_q1_1 = """
SELECT
    collision_year AS Year,
    COUNT(*) AS Total,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) || '%' AS Percentage
FROM accidents
GROUP BY collision_year
ORDER BY collision_year;
"""

stats_q1_2 = """
SELECT
    CASE
        WHEN month IN (12, 1, 2) THEN 'Winter'
        WHEN month IN (3, 4, 5) THEN 'Spring'
        WHEN month IN (6, 7, 8) THEN 'Summer'
        ELSE 'Autumn'
    END AS Season,
    COUNT(*) AS Total
FROM accidents
GROUP BY Season
ORDER BY Total DESC;
"""

stats_q1_3 = """
SELECT
    CASE
        WHEN hour BETWEEN 7 AND 9 THEN 'Morning Rush'
        WHEN hour BETWEEN 16 AND 19 THEN 'Evening Rush'
        ELSE 'Off-Peak'
    END AS "Time Segment",
    COUNT(*) AS Total,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) || '%' AS Contribution
FROM accidents
GROUP BY "Time Segment"
ORDER BY Total DESC;
"""

stats_q2_1 = "SELECT speed_limit AS \"Speed Limit\", COUNT(*) AS Total FROM accidents GROUP BY 1 ORDER BY 2 DESC LIMIT 5;"
stats_q2_2 = "SELECT road_type_label AS \"Road Type\", collision_severity_label AS Severity, COUNT(*) AS Total FROM accidents GROUP BY 1, 2 ORDER BY 3 DESC LIMIT 5;"

stats_q3_1 = "SELECT weather_conditions_label AS Weather, light_conditions_label AS Lighting, COUNT(*) AS Total FROM accidents GROUP BY 1, 2 ORDER BY 3 DESC LIMIT 5;"
stats_q3_2 = "SELECT road_type_label AS \"Road Type\", weather_conditions_label AS Weather, hour AS Hour, COUNT(*) AS Total FROM accidents GROUP BY 1, 2, 3 ORDER BY 4 DESC LIMIT 5;"

## 4. Thực thi, Lưu tệp `sql/queries.sql` và Xuất kết quả ra `report/table_q*.csv`

Đoạn mã tự động ghi nhận file `sql/queries.sql` và xuất các bảng kết quả ra thư mục `report/table_q*.csv` phục vụ đưa vào báo cáo chính thức.

In [ ]:
SQL_SCRIPT = """
-- Q1: Thống kê số lượng báo cáo theo năm và nhóm UAS
SELECT year, is_uas, COUNT(*) AS n FROM r GROUP BY 1, 2 ORDER BY 1, 2;

-- Q2: Tỷ trọng giai đoạn bay xảy ra sự cố
SELECT is_uas, flight_phase, COUNT(*) AS n, COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY is_uas) AS pct FROM r GROUP BY 1, 2 ORDER BY 1, pct DESC;

-- Q3: Nhãn sự cố và độ dài mô tả văn bản
SELECT y, COUNT(*) AS n, AVG(length(narrative)) AS len FROM r GROUP BY 1 ORDER BY n DESC;

-- Q4: Phân tích rủi ro vùng trời và thời tiết
SELECT airspace, weather, COUNT(*) AS total_incidents FROM r GROUP BY 1, 2 ORDER BY total_incidents DESC LIMIT 10;
"""

with open('sql/queries.sql', 'w', encoding='utf-8') as f:
    f.write(SQL_SCRIPT.strip())

print("Đã lưu script SQL vào: sql/queries.sql")

queries = [q.strip() for q in SQL_SCRIPT.split(';') if q.strip()]
for idx, q in enumerate(queries, 1):
    df_res = con.execute(q).df()
    output_file = f"report/table_q{idx}.csv"
    df_res.to_csv(output_file, index=False)
    print(f"Đã lưu kết quả Query {idx} vào: {output_file}")

## 5. Nhận xét mẫu (Interpretation / Human Delta) Cho Báo cáo

- **Nhận xét Query 1.1 & 1.2 (RQ1):**
  > Số lượng báo cáo sự cố UAS có xu hướng gia tăng theo các năm, tập trung nhiều nhất ở các giai đoạn bay "Cruise" và "Initial Approach". Việc áp dụng Window Function cho thấy tỷ lệ sự cố UAS trong giai đoạn tiếp cận cao hơn so với máy bay thông thường, phản ánh rủi ro mất kiểm soát khi hoạt động gần mặt đất.

- **Nhận xét Query 1.3 (RQ2 & RQ3):**
  > Đoạn văn bản Narrative của các sự cố liên quan đến "Deviation" và "Conflict" có độ dài trung bình lớn nhất (> 1,650 ký tự), chứng tỏ các sự cố phức tạp đòi hỏi báo cáo chi tiết, làm cơ sở quan trọng cho các mô hình tóm tắt nhỏ (SLMs) xử lý.